# AnyCam — 06b · Step 9 (E5): cross-validation, folds 3, 4

Step 7 used ONE split. Here the frozen final settings are repeated on the plot-grouped cross-validation folds
(test plots never seen in training), so the paper can report mean ± std.
Per fold: **204-band baseline** (+ test on all cameras), **SVCT wavelength stem**, **SVCT interp**.
Both GPUs are used at once (GPU0: baseline + stem, GPU1: interp).

Time: ~3.5-4 h. The other notebook is **06a (folds 0, 1, 2)**.

Setup: **HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All** (only once!).
GitHub: **ShahnawazQ007/one-model-many-camera**, file `kaggle_runs/06b_step9_cv.ipynb`.

In [1]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

GPU: Tesla T4
data: /kaggle/input/competitions/HyperLeaf2024 | images: 2410


In [2]:
# all training for these folds (progress line every 10 min)
!python scripts/run_cv.py --data {DATA} --folds 3 4 --out runs/cv

2 GPU(s); folds [3, 4] seed 0; streams: [(0, ['baseline', 'svct']), (1, ['interp'])]
... 10 min | GPU0 baseline_s0_f3: running | GPU1 interp_s0_f3: running
... 20 min | GPU0 baseline_s0_f3: running | GPU1 interp_s0_f3: running
... 30 min | GPU0 baseline_s0_f3: running | GPU1 interp_s0_f3: running
... 40 min | GPU0 baseline_s0_f3: running | GPU1 interp_s0_f3: running
... 50 min | GPU0 baseline_s0_f3: running | GPU1 interp_s0_f3: running
... 60 min | GPU0 baseline_s0_f3: running | GPU1 interp_s0_f3: running
[GPU0] baseline_s0_f3 ok in 67 min
... 70 min | GPU0 baseline_eval_s0_f3: running | GPU1 interp_s0_f3: running
... 80 min | GPU0 baseline_eval_s0_f3: running | GPU1 interp_s0_f3: running
... 90 min | GPU0 baseline_eval_s0_f3: running | GPU1 interp_s0_f3: running
... 100 min | GPU0 baseline_eval_s0_f3: running | GPU1 interp_s0_f3: running
... 110 min | GPU0 baseline_eval_s0_f3: running | GPU1 interp_s0_f3: running
... 120 min | GPU0 baseline_eval_s0_f3: running | GPU1 interp_s0_f3: run

In [3]:
# summary of these folds + raw numbers (Claude reads them from the saved notebook)
!python scripts/summarize_cv.py --cv runs/cv --out runs/cv_summary
print('=== CV_PER_FOLD_CSV ==='); print(open('runs/cv_summary/cv_per_fold.csv').read())
print('=== CV_ALL_CAMERAS_CSV ==='); print(open('runs/cv_summary/cv_all_cameras.csv').read())

Mean (std over folds) of the mean over the 7 held-out camera profiles:
                                     cultivar_acc   Fertilizer_R2  GrainWeight_R2       PhiPS2_R2  n_folds
method                 condition                                                                          
204-band baseline      drift        0.342 (0.043)  -0.311 (0.318)  -1.018 (0.946)  -0.119 (0.252)        2
                       drift+radio  0.334 (0.080)  -0.583 (0.493)  -1.469 (1.290)  -0.212 (0.266)        2
                       nominal      0.332 (0.048)  -0.229 (0.285)  -0.943 (0.989)  -0.131 (0.282)        2
                       radiometric  0.328 (0.077)  -0.572 (0.639)  -1.477 (1.525)  -0.195 (0.264)        2
SVCT (interp)          drift        0.744 (0.061)   0.528 (0.091)   0.400 (0.067)   0.294 (0.069)        2
                       drift+radio  0.722 (0.062)   0.497 (0.087)   0.345 (0.093)   0.257 (0.050)        2
                       nominal      0.748 (0.058)   0.542 (0.094)   0.413

## Done
When the version finishes: open it → **Save to GitHub / Quick Save** → repository **ShahnawazQ007/one-model-many-camera**, file `kaggle_runs/06b_step9_cv.ipynb`. Tell Claude "done".